## Resumen
Este notebook replica en Python el enfoque de Andres para construir bins (new_bins) y tabla de Lorenz (lorenz_table), y valida como cambia la media al pasar de microdatos a 1000 bins frente a los benchmarks de PIP y GlobalDist.

1: Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

2: Paths

In [ ]:
ROOT = Path(".").resolve()
INPUT = ROOT / "01-input"
OUT = ROOT / "work"

mwi_path = INPUT / "country" / "MWI_1997.dta"
pip_bins_path = INPUT / "pip_1kbins.dta"
global_bins_path = INPUT / "GlobalDist1000bins_1990_2026_20260324_2021_01_02_PROD.dta"

mwi_path, pip_bins_path, global_bins_path

(PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/country/MWI_1997.dta'),
 PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/pip_1kbins.dta'),
 PosixPath('/Users/danielaavayu/Desktop/World Bank/Bottom Coding/01-input/GlobalDist1000bins_1990_2026_20260324_2021_01_02_PROD.dta'))

3: Load PIP And Global-Bin Benchmarks

In [ ]:
pip_bins = pd.read_stata(pip_bins_path, convert_categoricals=False)
pip_bins["year"] = pip_bins["year"].astype(int)

mwi_pip_row = pip_bins[
    (pip_bins["country_code"] == "MWI")
    & (pip_bins["year"] == 1997)
].iloc[0]

pip_mean = mwi_pip_row["mean"]
global_1kbins_mean = mwi_pip_row["mean_1kbins"]

pd.DataFrame({
    "source": [
        "PIP mean",
        "Global 1000-bin mean from delivered file",
    ],
    "mean": [
        pip_mean,
        global_1kbins_mean,
    ],
})

,source,mean
0,PIP mean,5.724018
1,Global 1000-bin mean from delivered file,4.865801


4: Load Malawi Microdata

In [ ]:
mwi_raw = pd.read_stata(mwi_path, convert_categoricals=False)

mwi_raw.shape, mwi_raw.columns.tolist()

((10698, 41),
 ['countrycode',
  'hhid',
  'welfare',
  'urban',
  'hsize',
  'wta_hh',
  'weight',
  'fdpindex',
  'nfdpindex',
  'pc_hhdr',
  'subnatid',
  'code',
  'year',
  'survname',
  'cpi_domain',
  'cpi_domain_value',
  'cpi2021_unadj',
  'cpi2017_unadj',
  'cpi2011_unadj',
  'cpi2011',
  'cpi2017',
  'cpi2021',
  'icp2011',
  'icp2017',
  'icp2021',
  'comparability',
  'cpi_replication',
  'cpi_domain_var',
  'icpreplication',
  'icpdomain_var',
  'icpdomain_value',
  'comparable',
  'cpi2005',
  'icp2005',
  'cpi_data_level',
  'cpi2011_SM26',
  'cpi2017_SM26',
  'cpi2021_SM26',
  'icp2011_SM26',
  'icp2017_SM26',
  'icp2021_SM26'])

5: Prepare dt

In [ ]:
cpi2021 = mwi_raw["cpi2021"].dropna().iloc[0]
icp2021 = mwi_raw["icp2021"].dropna().iloc[0]

dt = mwi_raw.copy()

dt["welfare_ppp_day"] = (
    dt["welfare"] / (cpi2021 * icp2021 * 365)
)

dt = dt[["welfare_ppp_day", "weight"]].dropna().copy()
dt = dt[dt["weight"] > 0].copy()

# Match names expected by lorenz_table()
dt = dt.rename(columns={"welfare_ppp_day": "welfare"})

# Boss functions expect reporting_level
dt["reporting_level"] = "national"

dt[["welfare", "weight", "reporting_level"]].head()

,welfare,weight,reporting_level
0,3.189432,187.221756,national
1,4.211926,374.443512,national
2,3.025582,748.887024,national
3,1.666500,"1,310.552246",national
4,2.464965,374.443512,national


6: Check Input Mean

In [ ]:
direct_microdata_mean = np.average(
    dt["welfare"],
    weights=dt["weight"],
)

pd.DataFrame({
    "source": [
        "Direct microdata weighted mean",
        "PIP mean",
        "Global 1000-bin mean",
    ],
    "mean": [
        direct_microdata_mean,
        pip_mean,
        global_1kbins_mean,
    ],
})

,source,mean
0,Direct microdata weighted mean,5.779648
1,PIP mean,5.724018
2,Global 1000-bin mean,4.865801


7: Translate new_bins() From duplicate_households.R

Logic: 
Remove missing welfare or weights.
Sort observations from poorest to richest.
Compute total weighted population.
Divide total population by 1000.
Fill bin 1 with the poorest 0.1% of the weighted population.
Fill bin 2 with the next 0.1%.
Continue until bin 1000.
If one observation’s weight crosses a bin boundary, split that observation across bins.

In [ ]:
def new_bins(welfare, weight, nbins=1000, tolerance=1e-6, ids=None):
    welfare = np.asarray(welfare, dtype=float)
    weight = np.asarray(weight, dtype=float)

    valid = ~np.isnan(welfare) & ~np.isnan(weight)
    welfare = welfare[valid]
    weight = weight[valid]

    if ids is None:
        ids = np.arange(len(welfare))
    else:
        ids = np.asarray(ids)[valid]

    order = np.argsort(welfare)
    welfare = welfare[order]
    weight = weight[order]
    ids = ids[order]

    total_weight = weight.sum()
    bin_size = total_weight / nbins

    out_rows = []

    cur_bin = 1
    cur_weight = 0.0

    for id_i, w, wt in zip(ids, welfare, weight):
        while wt > 0 and cur_bin <= nbins:
            room = bin_size - cur_weight

            if abs(room) < tolerance:
                cur_bin += 1
                cur_weight = 0.0
                continue

            take = min(wt, room)

            out_rows.append({
                "id": id_i,
                "bin": cur_bin,
                "weight": take,
                "welfare": w,
            })

            wt -= take
            cur_weight += take

            if cur_weight >= bin_size - tolerance:
                cur_bin += 1
                cur_weight = 0.0

    return pd.DataFrame(out_rows)

8: Translate lorenz_table() From duplicate_households.R

This function takes the expanded bin assignments from new_bins() and collapses them into one row per bin.

In [ ]:
def lorenz_table(df, nq=1000, tolerance=1e-6):
    d = df.copy()

    no_dl = d["reporting_level"].nunique()

    if no_dl > 1:
        d2 = d.copy()
        d2["reporting_level"] = "national"
        d = pd.concat([d, d2], ignore_index=True)

    pieces = []

    for reporting_level, sub in d.groupby("reporting_level", observed=True):
        sub = sub.sort_values("welfare").reset_index(drop=True)
        sub["id"] = np.arange(len(sub))

        binned = new_bins(
            welfare=sub["welfare"].to_numpy(),
            weight=sub["weight"].to_numpy(),
            ids=sub["id"].to_numpy(),
            nbins=nq,
            tolerance=tolerance,
        )

        binned["reporting_level"] = reporting_level
        pieces.append(binned)

    expanded = pd.concat(pieces, ignore_index=True)

    expanded["wt_welfare"] = expanded["welfare"] * expanded["weight"]

    totals = (
        expanded.groupby("reporting_level", observed=True)
        .agg(
            tot_pop=("weight", "sum"),
            tot_wlf=("wt_welfare", "sum"),
        )
        .reset_index()
    )

    expanded = expanded.merge(totals, on="reporting_level", how="left")

    expanded["pop_share"] = expanded["weight"] / expanded["tot_pop"]
    expanded["welfare_share"] = expanded["wt_welfare"] / expanded["tot_wlf"]

    lt = (
        expanded.groupby(["reporting_level", "bin"], observed=True)
        .apply(lambda g: pd.Series({
            "avg_welfare": np.average(g["welfare"], weights=g["weight"]),
            "pop_share": g["pop_share"].sum(),
            "welfare_share": g["welfare_share"].sum(),
            "quantile": g["welfare"].max(),
            "pop": g["weight"].sum(),
        }))
        .reset_index()
        .sort_values(["reporting_level", "bin"])
    )

    return lt

9: Run  Methodology

In [ ]:
nq = 1000

lt = lorenz_table(dt, nq=nq)

# Match boss's censoring step.
lt.loc[lt["bin"] >= nq, "quantile"] = np.nan

lt.head(), lt.tail()

(  reporting_level  bin  avg_welfare  pop_share  welfare_share  quantile  \
 0        national    1     0.300561   0.001000       0.000052  0.346217   
 1        national    2     0.355088   0.001000       0.000061  0.377876   
 2        national    3     0.395883   0.001000       0.000068  0.405986   
 3        national    4     0.418235   0.001000       0.000072  0.430491   
 4        national    5     0.438012   0.001000       0.000076  0.454097   
 
            pop  
 0 9,666.648092  
 1 9,666.648092  
 2 9,666.648092  
 3 9,666.648092  
 4 9,666.648092  ,
     reporting_level   bin  avg_welfare  pop_share  welfare_share  quantile  \
 995        national   996    47.630438   0.001000       0.008241 52.280365   
 996        national   997    57.773889   0.001000       0.009996 66.554596   
 997        national   998    71.648409   0.001000       0.012397 76.766281   
 998        national   999    84.228571   0.001000       0.014573 98.250542   
 999        national  1000 2,172.69697

10: Compute Mean From 1000 Bins

In [ ]:
mean_from_boss_style_bins = np.average(
    lt["avg_welfare"],
    weights=lt["pop"],
)

pd.DataFrame({
    "source": [
        "Direct microdata weighted mean",
        "Mean from boss-style 1000 bins",
        "PIP mean",
        "Global 1000-bin mean",
    ],
    "mean": [
        direct_microdata_mean,
        mean_from_boss_style_bins,
        pip_mean,
        global_1kbins_mean,
    ],
    "difference_vs_global_1kbins": [
        direct_microdata_mean - global_1kbins_mean,
        mean_from_boss_style_bins - global_1kbins_mean,
        pip_mean - global_1kbins_mean,
        0,
    ],
})

,source,mean,difference_vs_global_1kbins
0,Direct microdata weighted mean,5.779648,0.913847
1,Mean from boss-style 1000 bins,5.779648,0.913847
2,PIP mean,5.724018,0.858217
3,Global 1000-bin mean,4.865801,0.000000


In [ ]:
lt["pop"].describe()

count   1,000.000000
mean    9,666.648092
std         0.000000
min     9,666.648092
25%     9,666.648092
50%     9,666.648092
75%     9,666.648092
max     9,666.648092
Name: pop, dtype: float64